# Module 13 — Multi-agent systems with the A2A protocol

> Part of the **"Develop & Deploy AI Agents on Azure with LangChain, Python and Foundry"** course.

In Module 12 we kept all sub-agents **inside one Python process**. That's perfect for prototyping — but in production agents are often built by **different teams**, in **different languages**, and they need to **discover and talk to each other across the network**.

That's exactly what the **Agent-to-Agent (A2A) protocol** does.

## 🎯 Learning objectives

1. Understand what **A2A** is and how it complements MCP.
2. Read an **agent card** (`/.well-known/agent.json`).
3. Expose a LangChain agent as an **A2A server**.
4. Call a remote agent from another agent using the **`a2a` Python SDK**.
5. Pick A2A vs MCP vs sub-agent depending on your topology.

## 🧠 MCP vs A2A — they are not competitors

| Question                              | MCP                                    | A2A                                     |
| ------------------------------------- | -------------------------------------- | --------------------------------------- |
| Who's on the other side?              | A **tool server** (no autonomy)        | Another **agent** (has its own loop)    |
| Protocol shape                        | JSON-RPC: `list_tools`, `call_tool`    | HTTP: `tasks/send`, `tasks/get`, …      |
| Discovery                             | Configured client side                 | `/.well-known/agent.json` — self-describing |
| Streaming                             | Yes (SSE)                              | Yes (SSE)                               |
| Stateful?                             | No (each call independent)             | Yes — A2A defines **tasks** with status |

Mental model:

```
        ┌─────────────────────────────────────┐
        │            Supervisor agent         │
        ├─────────────────────────────────────┤
        │  uses MCP servers  → for tools      │
        │  uses A2A peers    → for agents     │
        └─────────────────────────────────────┘
```

## 📇 The agent card

Every A2A-compliant agent serves an **agent card** at `/.well-known/agent.json`. It's the agent's business card:

```jsonc
{
  "name": "web-research-agent",
  "description": "Researches a topic on the public web and returns a summary.",
  "url": "https://research.contoso.com",
  "capabilities": { "streaming": true, "pushNotifications": false },
  "skills": [
    { "id": "research", "name": "Web research", "description": "..." }
  ]
}
```

A supervisor agent can **fetch this card at runtime**, decide whether the peer can help, and call it. No SDK lock-in — anyone speaking A2A can join the conversation.

## 🛠 Install the SDK

In [ ]:
%pip install "a2a-sdk[client,server]" uvicorn langchain langchain-openai langgraph

## 🖥️ Step 1 — Wrap a LangChain agent as an A2A server

Below is the skeleton — fill in your model endpoint and run the file in a separate terminal:

```bash
python research_a2a_server.py
```

In [ ]:
from pathlib import Path

Path("research_a2a_server.py").write_text('''
from a2a.server import A2AServer, AgentCard, Skill
from a2a.types import Task, TextPart
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
import os, uvicorn

@tool
def fake_search(query: str) -> str:
    """Pretend to search the web. Replace with a real tool later."""
    return f"Top result for {query!r}: Azure docs say…"

model = ChatOpenAI(
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["API_KEY"],
    model=os.environ["MODEL"],
)
agent = create_agent(model=model, tools=[fake_search])

card = AgentCard(
    name="web-research-agent",
    description="Researches a topic and summarises it.",
    url="http://localhost:8080",
    skills=[Skill(id="research", name="Web research",
                  description="Research a topic on the web.")],
)

async def handle(task: Task) -> Task:
    text = task.message.parts[0].text
    result = await agent.ainvoke({"messages": [{"role": "user", "content": text}]})
    task.add_message("agent", [TextPart(text=result["messages"][-1].content)])
    task.complete()
    return task

if __name__ == "__main__":
    uvicorn.run(A2AServer(card, handler=handle), host="0.0.0.0", port=8080)
''')
print("Wrote research_a2a_server.py — run it in a separate terminal.")

## 📞 Step 2 — Call the remote agent from a client

Now from the supervisor (this notebook) we fetch the agent card and send a task.

In [ ]:
from a2a.client import A2AClient

client = await A2AClient.from_url("http://localhost:8080")
print("Peer card:", client.card.name, "-", client.card.description)

task = await client.send_task("What are the cheapest GPU SKUs on Azure Container Apps?")
async for update in client.stream(task.id):
    print(update)

## 🏗️ Step 3 — Wire the remote agent as a tool of the supervisor

Just like in Module 12 we expose the sub-agent as `@tool`. The only difference: the implementation now **calls the network** instead of an in-process function.

In [ ]:
from langchain_core.tools import tool

@tool
async def ask_research_agent(question: str) -> str:
    """Delegate a research question to the remote web-research agent."""
    task = await client.send_task(question)
    final = await client.wait_for(task.id)
    return final.message.parts[0].text

# Pass [ask_research_agent] to create_agent(...) as you did in Module 12.

## ✅ Recap

* MCP gives an agent **tools**. A2A gives it **peers**.
* Every A2A agent advertises itself with an **agent card** under `/.well-known/agent.json`.
* A2A peers can be hosted **anywhere** — Container Apps, App Service, on-prem — and written in **any language**.

Combine **sub-agents** (Module 12), **MCP** (Module 8) and **A2A** (this module) to build agent ecosystems.